# Hacking Forums — 03: Embeddings y Perfilado

Representación vectorial de usuarios, pivoting cross-foro y atribución de identidad.
Estrategias: embed_users (A) + centroids sampled top-50 más largos (C).
5 foros · ~21K usuarios embed_users · ~11K usuarios con centroides.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
from difflib import SequenceMatcher

from src.utils import load_or_compute, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"RESULTS_DIR  : {RESULTS_DIR}")

In [ ]:
HF_RESULTS = RESULTS_DIR / 'hacking_forums'
HF_RESULTS.mkdir(parents=True, exist_ok=True)


def load_latest(pattern):
    matches = sorted(HF_RESULTS.glob(pattern))
    if not matches:
        print(f"  [WARN] no files match: {pattern}")
        return None
    path = matches[-1]
    print(f"  Cargando: {path.name}")
    return dict(np.load(path, allow_pickle=True))


# Load embed_users (Strategy A)
emb_a = load_latest('hacking_forums_user_embeddings.npz')
if emb_a is not None:
    user_ids_a = emb_a['user_ids'].tolist()
    vectors_a  = emb_a['vectors']
    print(f"A — embed_users : {len(user_ids_a):,} usuarios, dim={vectors_a.shape[1]}")
else:
    user_ids_a = []
    vectors_a  = np.empty((0, 4096), dtype=np.float32)

# Load centroids sampled (Strategy C)
emb_c = load_latest('hf_centroids_sampled_*.npz')
if emb_c is not None:
    user_ids_c = emb_c['user_ids'].tolist()
    vectors_c  = emb_c['vectors']
    print(f"C — centroides  : {len(user_ids_c):,} usuarios, dim={vectors_c.shape[1]}")
else:
    user_ids_c = []
    vectors_c  = np.empty((0, 4096), dtype=np.float32)

# Load users table for handle lookup
USERS_PARQUET = HF_RESULTS / 'users_clean.parquet'
if USERS_PARQUET.exists():
    users_df = pd.read_parquet(USERS_PARQUET)
    print(f"users_clean     : {len(users_df):,} filas, foros: {users_df['forum'].nunique() if 'forum' in users_df.columns else '?'}")
else:
    print(f"  [WARN] {USERS_PARQUET.name} no encontrado — pivoting deshabilitado")
    users_df = pd.DataFrame()

## Sección 1 — Embeddings: estrategias A vs C

Comparamos si ambas estrategias preservan la misma estructura de ranking de similitud entre usuarios.
Usamos la **correlación de Spearman** sobre las similitudes coseno pairwise del subconjunto de usuarios
presentes en ambas estrategias.

- **A (embed_users)**: 1 vector por usuario concatenando todos sus posts. Captura el perfil global.
- **C (centroides muestreados)**: media de los embeddings individuales de los top-50 posts más largos. Más sensible a variabilidad interna.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

spearman_rho = None

if len(user_ids_a) > 0 and len(user_ids_c) > 0:
    set_a = set(user_ids_a)
    set_c = set(user_ids_c)
    common = sorted(set_a & set_c)
    print(f"Usuarios en A: {len(user_ids_a):,}")
    print(f"Usuarios en C: {len(user_ids_c):,}")
    print(f"Intersección : {len(common):,}")

    if len(common) >= 50:
        # Cap at 2000 to keep computation tractable
        sample = common[:2000]

        idx_a = {uid: i for i, uid in enumerate(user_ids_a)}
        idx_c = {uid: i for i, uid in enumerate(user_ids_c)}

        vecs_a_sub = vectors_a[[idx_a[u] for u in sample]]
        vecs_c_sub = vectors_c[[idx_c[u] for u in sample]]

        sim_a = cosine_similarity(vecs_a_sub)
        sim_c = cosine_similarity(vecs_c_sub)

        # Upper triangle (excluding diagonal)
        triu_idx = np.triu_indices(len(sample), k=1)
        flat_a = sim_a[triu_idx]
        flat_c = sim_c[triu_idx]

        rho, pval = spearmanr(flat_a, flat_c)
        spearman_rho = rho
        print(f"\nSpearman ρ = {rho:.4f}  (p = {pval:.2e})")
        if abs(rho) >= 0.7:
            print("→ Correlación fuerte: ambas estrategias preservan la estructura de similitud.")
        elif abs(rho) >= 0.4:
            print("→ Correlación moderada: las estrategias coinciden en tendencia pero difieren en magnitud.")
        else:
            print("→ Correlación débil: las estrategias capturan señales distintas.")
    else:
        print("  Intersección insuficiente para comparación.")
else:
    print("  Embeddings no disponibles — omitiendo comparación Spearman.")

## Sección 2 — UMAP + clusters

Reducimos a 2D con UMAP para visualizar la distribución de estilos de escritura.
HDBSCAN detecta clusters densos sin asumir forma ni número de grupos.
Usamos la estrategia C (centroides) por ser más compacta y más limpia para clustering.

In [ ]:
import umap as umap_lib
import hdbscan as hdbscan_lib

hdb_labels = np.array([])
coords_c_2d = None

if len(vectors_c) >= 10:
    print("Ejecutando UMAP sobre estrategia C...")
    reducer = umap_lib.UMAP(n_components=2, random_state=42,
                             n_neighbors=min(15, len(vectors_c) - 1),
                             metric='cosine')
    coords_c_2d = reducer.fit_transform(vectors_c)
    print(f"UMAP completado: {coords_c_2d.shape}")

    print("Ejecutando HDBSCAN...")
    clusterer = hdbscan_lib.HDBSCAN(min_cluster_size=10, min_samples=5,
                                     metric='euclidean')
    hdb_labels = clusterer.fit_predict(coords_c_2d)
    n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    n_noise    = (hdb_labels == -1).sum()
    print(f"Clusters: {n_clusters}  |  Ruido: {n_noise:,} ({n_noise/len(hdb_labels)*100:.1f}%)")

    # Build forum prefix from user_ids if available
    forum_labels = []
    for uid in user_ids_c:
        uid_str = str(uid)
        if '_' in uid_str:
            forum_labels.append(uid_str.split('_')[0][:12])
        else:
            forum_labels.append('unknown')

    fig = go.Figure()
    palette = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
               '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

    unique_clusters = sorted(set(hdb_labels))
    for cl in unique_clusters:
        mask = hdb_labels == cl
        color = '#444' if cl == -1 else palette[cl % len(palette)]
        name  = 'Ruido' if cl == -1 else f'Cluster {cl}'
        fig.add_trace(go.Scattergl(
            x=coords_c_2d[mask, 0],
            y=coords_c_2d[mask, 1],
            mode='markers',
            marker=dict(size=3, color=color, opacity=0.6 if cl != -1 else 0.2),
            text=[forum_labels[i] for i in np.where(mask)[0]],
            hovertemplate='%{text}<extra>' + name + '</extra>',
            name=name,
        ))

    fig.update_layout(
        title=f'UMAP + HDBSCAN — Estrategia C (centroides muestreados)<br>'
              f'{n_clusters} clusters · {n_noise:,} ruido',
        template='plotly_dark', height=600, width=950,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
    fig.show()
else:
    print("Embeddings C no disponibles — UMAP omitido.")

In [ ]:
# Cluster composition: how many forums does each cluster mix?
if len(hdb_labels) > 0 and len(forum_labels) > 0:
    cluster_comp = pd.DataFrame({
        'user_id': user_ids_c,
        'forum':   forum_labels,
        'cluster': hdb_labels,
    })
    cluster_comp = cluster_comp[cluster_comp['cluster'] != -1]

    comp_table = (
        cluster_comp
        .groupby('cluster')
        .agg(
            n_users=('user_id', 'count'),
            n_forums=('forum', 'nunique'),
            forums=('forum', lambda x: ', '.join(sorted(x.unique())))
        )
        .sort_values('n_users', ascending=False)
        .reset_index()
    )

    mixed = comp_table[comp_table['n_forums'] > 1]
    print(f"Clusters totales (sin ruido): {len(comp_table)}")
    print(f"Clusters que mezclan foros  : {len(mixed)}")
    print()
    print("Top clusters por tamaño:")
    print(comp_table.head(20).to_string(index=False))

    if len(mixed) > 0:
        print("\nClusters cross-foro (mezclan ≥2 foros):")
        print(mixed.to_string(index=False))
else:
    print("No hay datos de clusters para analizar.")

## Sección 3 — Pivoting cross-foro

Buscamos usuarios que aparecen en 2 o más foros con el **mismo handle** (exacto, case-insensitive).
Esto es el primer indicador de misma identidad cross-foro, aunque no es concluyente por sí solo.

In [ ]:
exact_matches_df = pd.DataFrame()

if not users_df.empty and 'username' in users_df.columns and 'forum' in users_df.columns:
    users_df['handle_norm'] = users_df['username'].astype(str).str.lower().str.strip()

    handle_forums = (
        users_df
        .groupby('handle_norm')['forum']
        .agg(lambda x: sorted(set(x)))
        .reset_index()
        .rename(columns={'forum': 'forums'})
    )
    handle_forums['n_forums'] = handle_forums['forums'].apply(len)
    handle_forums['forums_str'] = handle_forums['forums'].apply(lambda x: ', '.join(x))

    cross_handles = handle_forums[handle_forums['n_forums'] >= 2].sort_values('n_forums', ascending=False)
    exact_matches_df = cross_handles.copy()

    print(f"Handles únicos totales      : {len(handle_forums):,}")
    print(f"Handles en ≥2 foros (exacto): {len(cross_handles):,}")
    print()
    print("Top 30 handles cross-foro:")
    print(
        cross_handles[['handle_norm', 'n_forums', 'forums_str']]
        .head(30)
        .to_string(index=False)
    )
else:
    print("users_clean.parquet no disponible o sin columnas 'username'/'forum'.")
    print("Pivoting cross-foro omitido.")

In [ ]:
fuzzy_results_df = pd.DataFrame()

if not users_df.empty and 'handle_norm' in users_df.columns and 'forum' in users_df.columns:
    try:
        from rapidfuzz import fuzz, process
        USE_RAPIDFUZZ = True
    except ImportError:
        USE_RAPIDFUZZ = False
        print("rapidfuzz no instalado — usando SequenceMatcher (más lento)")

    FUZZY_THRESHOLD = 85  # %

    # Build per-forum handle sets (sample to keep it fast)
    forum_handles = {}
    for forum, grp in users_df.groupby('forum'):
        handles = grp['handle_norm'].dropna().unique().tolist()
        forum_handles[forum] = handles

    forums_list = list(forum_handles.keys())
    print(f"Foros disponibles: {forums_list}")

    fuzzy_rows = []
    for i, forum_a in enumerate(forums_list):
        handles_a = forum_handles[forum_a][:500]   # cap per forum
        for forum_b in forums_list[i+1:]:
            handles_b = forum_handles[forum_b]

            if USE_RAPIDFUZZ:
                for ha in handles_a:
                    res = process.extractOne(ha, handles_b,
                                             scorer=fuzz.ratio,
                                             score_cutoff=FUZZY_THRESHOLD)
                    if res:
                        hb, score, _ = res
                        if ha != hb:
                            fuzzy_rows.append({
                                'handle_a': ha, 'forum_a': forum_a,
                                'handle_b': hb, 'forum_b': forum_b,
                                'similarity': round(score / 100, 3),
                            })
            else:
                for ha in handles_a:
                    for hb in handles_b[:500]:
                        if ha == hb:
                            continue
                        ratio = SequenceMatcher(None, ha, hb).ratio()
                        if ratio >= FUZZY_THRESHOLD / 100:
                            fuzzy_rows.append({
                                'handle_a': ha, 'forum_a': forum_a,
                                'handle_b': hb, 'forum_b': forum_b,
                                'similarity': round(ratio, 3),
                            })

    if fuzzy_rows:
        fuzzy_results_df = (
            pd.DataFrame(fuzzy_rows)
            .drop_duplicates()
            .sort_values('similarity', ascending=False)
        )
        print(f"\nMatches fuzzy (≥{FUZZY_THRESHOLD}%): {len(fuzzy_results_df):,}")
        print(fuzzy_results_df.head(30).to_string(index=False))
    else:
        print("Sin matches fuzzy encontrados.")
else:
    print("users_clean.parquet no disponible — fuzzy matching omitido.")

In [ ]:
attribution_candidates = pd.DataFrame()

# Combine exact + fuzzy handle candidates
candidate_pairs = []

if not exact_matches_df.empty:
    for _, row in exact_matches_df.iterrows():
        forums = row['forums']
        for fi, fa in enumerate(forums):
            for fb in forums[fi+1:]:
                candidate_pairs.append({
                    'handle_a': row['handle_norm'], 'forum_a': fa,
                    'handle_b': row['handle_norm'], 'forum_b': fb,
                    'handle_sim': 1.0,
                    'match_type': 'exact',
                })

if not fuzzy_results_df.empty:
    for _, row in fuzzy_results_df.head(100).iterrows():
        candidate_pairs.append({
            'handle_a': row['handle_a'], 'forum_a': row['forum_a'],
            'handle_b': row['handle_b'], 'forum_b': row['forum_b'],
            'handle_sim': row['similarity'],
            'match_type': 'fuzzy',
        })

if candidate_pairs and len(vectors_c) > 0 and not users_df.empty:
    # Build user_id → centroid index map
    uid_to_cidx = {uid: i for i, uid in enumerate(user_ids_c)}

    # Build handle+forum → user_id map
    if 'userid' in users_df.columns:
        hf_to_uid = {
            (row['handle_norm'], row['forum']): str(row['userid'])
            for _, row in users_df[['handle_norm', 'forum', 'userid']].dropna().iterrows()
        }
    else:
        hf_to_uid = {}

    enriched = []
    for pair in candidate_pairs:
        uid_a = hf_to_uid.get((pair['handle_a'], pair['forum_a']))
        uid_b = hf_to_uid.get((pair['handle_b'], pair['forum_b']))
        emb_sim = None
        if uid_a and uid_b and uid_a in uid_to_cidx and uid_b in uid_to_cidx:
            va = vectors_c[uid_to_cidx[uid_a]].reshape(1, -1)
            vb = vectors_c[uid_to_cidx[uid_b]].reshape(1, -1)
            emb_sim = float(cosine_similarity(va, vb)[0, 0])
        enriched.append({**pair, 'embedding_sim': emb_sim})

    attribution_candidates = (
        pd.DataFrame(enriched)
        .dropna(subset=['embedding_sim'])
        .sort_values('embedding_sim', ascending=False)
    )
    print(f"Candidatos con embedding sim: {len(attribution_candidates):,}")
    print(attribution_candidates.head(20).to_string(index=False))
else:
    print("Sin candidatos o embeddings C no disponibles.")

## Sección 4 — Estilometría: Burrows' Delta (cross-foro)

**Burrows' Delta** mide la distancia estilística entre autores usando las *n* palabras más frecuentes
del corpus, expresadas como z-scores:

$$\Delta(A, B) = \frac{1}{n} \sum_{i=1}^{n} |z_i^A - z_i^B|$$

**Delta bajo** → misma distribución de palabras función → candidato a mismo autor.

Extensión respecto al notebook 02: aquí corremos Delta **entre foros** (no solo OGUsers).
Tomamos los top-20 usuarios más activos de cada foro con posts disponibles,
extraemos las top-200 palabras función del corpus combinado,
y calculamos la matriz Delta completa cross-foro.

In [ ]:
# Re-use raw_forums if available in kernel, else load posts_clean.parquet
POSTS_PARQUET = HF_RESULTS / 'posts_clean.parquet'

stylo_df = pd.DataFrame()

if POSTS_PARQUET.exists():
    posts_clean = pd.read_parquet(POSTS_PARQUET)
    text_col = 'text' if 'text' in posts_clean.columns else (
        'pagetext' if 'pagetext' in posts_clean.columns else 'message'
    )
    posts_clean = posts_clean.dropna(subset=[text_col])
    posts_clean['text'] = posts_clean[text_col].astype(str)
    if 'forum' not in posts_clean.columns and 'source_forum' in posts_clean.columns:
        posts_clean['forum'] = posts_clean['source_forum']
    posts_clean['user_key'] = posts_clean['forum'].astype(str) + '_' + posts_clean['userid'].astype(str)
    stylo_df = posts_clean[['user_key', 'userid', 'forum', 'text']].copy()
    print(f"posts_clean.parquet: {len(stylo_df):,} posts, {stylo_df['user_key'].nunique():,} usuarios")
    print(f"Foros: {sorted(stylo_df['forum'].unique())}")
else:
    print(f"[WARN] {POSTS_PARQUET} no encontrado.")
    print("Ejecutá el paso de carga de datos de 02_hacking_forums primero,")
    print("o asegurate de que posts_clean.parquet exista en HF_RESULTS.")

In [ ]:
delta_corpus = {}
top_users_per_forum = {}

MIN_WORDS = 500
USERS_PER_FORUM = 20

if not stylo_df.empty:
    stylo_df = stylo_df[stylo_df['text'].str.strip().str.len() > 20]
    wc = stylo_df.groupby('user_key')['text'].apply(lambda x: len(' '.join(x).split()))
    eligible = wc[wc >= MIN_WORDS]
    print(f"Usuarios con ≥{MIN_WORDS} palabras: {len(eligible):,}")

    # Select top-N per forum
    forum_col = stylo_df[['user_key', 'forum']].drop_duplicates().set_index('user_key')['forum']
    for forum in stylo_df['forum'].unique():
        forum_users = [u for u in eligible.index if forum_col.get(u) == forum]
        top_n = eligible[forum_users].nlargest(USERS_PER_FORUM).index.tolist()
        top_users_per_forum[forum] = top_n
        if top_n:
            print(f"  {forum}: {len(top_n)} usuarios seleccionados")

    selected_users = [u for users in top_users_per_forum.values() for u in users]
    print(f"\nTotal usuarios para Delta: {len(selected_users)}")

    for uid in selected_users:
        texts = stylo_df[stylo_df['user_key'] == uid]['text'].tolist()
        delta_corpus[uid] = ' '.join(texts)
else:
    print("No hay posts para estilometría.")

In [ ]:
VOCAB_SIZE = 200
delta_df = pd.DataFrame()
z_matrix = pd.DataFrame()

if delta_corpus:
    all_words = ' '.join(delta_corpus.values()).lower().split()
    vocab = [w for w, _ in Counter(all_words).most_common(VOCAB_SIZE + 100)
             if re.match(r'^[a-z]{2,}$', w)][:VOCAB_SIZE]
    print(f"Vocabulario top-{VOCAB_SIZE}: {vocab[:10]}...")

    uids = list(delta_corpus.keys())
    freq_matrix = pd.DataFrame(index=uids, columns=vocab, dtype=float)
    for uid, text in delta_corpus.items():
        words = text.lower().split()
        total = max(len(words), 1)
        c = Counter(words)
        freq_matrix.loc[uid] = [c.get(w, 0) / total for w in vocab]

    z_matrix = (freq_matrix - freq_matrix.mean()) / freq_matrix.std().clip(lower=1e-9)
    print(f"Z-score matrix: {z_matrix.shape}")
else:
    print("Corpus vacío — omitiendo Delta.")

In [ ]:
if not z_matrix.empty:
    z_vals = z_matrix.values.astype(float)
    n = len(z_vals)
    delta_vals = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = float(np.mean(np.abs(z_vals[i] - z_vals[j])))
            delta_vals[i, j] = d
            delta_vals[j, i] = d

    uids = z_matrix.index.tolist()
    delta_df = pd.DataFrame(delta_vals, index=uids, columns=uids)
    print(f"Delta matrix: {delta_df.shape[0]}×{delta_df.shape[1]}")
    print(f"Rango: {delta_vals[delta_vals > 0].min():.3f} – {delta_vals.max():.3f}")

    # Top pairs (lowest delta = most similar)
    pairs = []
    forum_map = {u: f for f, us in top_users_per_forum.items() for u in us}
    for i in range(n):
        for j in range(i + 1, n):
            fa = forum_map.get(uids[i], '?')
            fb = forum_map.get(uids[j], '?')
            pairs.append({
                'user_a': uids[i], 'forum_a': fa,
                'user_b': uids[j], 'forum_b': fb,
                'cross_forum': fa != fb,
                'delta': round(delta_vals[i, j], 4),
            })
    delta_pairs_df = pd.DataFrame(pairs).sort_values('delta')

    print("\nTop 20 pares delta más bajo (todos):")
    print(delta_pairs_df.head(20)[['user_a','forum_a','user_b','forum_b','cross_forum','delta']].to_string(index=False))

    cross_pairs = delta_pairs_df[delta_pairs_df['cross_forum']]
    print(f"\nPares CROSS-FORO con delta más bajo (candidatos inter-foro):")
    print(cross_pairs.head(20)[['user_a','forum_a','user_b','forum_b','delta']].to_string(index=False))
else:
    delta_pairs_df = pd.DataFrame()
    print("Delta matrix no disponible.")

In [ ]:
if not delta_df.empty:
    # Use short labels: forum_prefix + user_id suffix
    def short_label(uid):
        parts = uid.split('_')
        forum_prefix = parts[0][:8] if parts else uid
        uid_suffix   = parts[-1][-4:] if len(parts) > 1 else ''
        return f"{forum_prefix}:{uid_suffix}"

    sub = delta_df.copy()
    sub.index   = [short_label(u) for u in delta_df.index]
    sub.columns = [short_label(u) for u in delta_df.columns]

    fig, ax = plt.subplots(figsize=(14, 11))
    mask = np.eye(len(sub), dtype=bool)
    sns.heatmap(sub, ax=ax, cmap='YlOrRd_r', mask=mask,
                linewidths=0.2, linecolor='#333',
                cbar_kws={'label': "Burrows' Delta (menor = más similar)"})
    ax.set_title(
        "Burrows' Delta — cross-foro\n"
        "Celdas frías (oscuras) = estilos similares → candidato a mismo actor",
        fontsize=12
    )
    ax.tick_params(axis='x', rotation=60, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)
    plt.tight_layout()
    out_path = HF_RESULTS / 'hf_delta_crossforum_heatmap.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardado: {out_path}")
else:
    print("Delta matrix no disponible.")

## Sección 5 — Señal combinada: handle + estilometría + embedding

Combinamos tres señales independientes para producir candidatos de atribución cross-foro robustos:

| Señal | Rango | Interpretación |
|---|---|---|
| `handle_sim` | 0–1 | 1 = mismo handle exacto |
| `delta` | 0–∞ | Más bajo = estilo más similar |
| `embedding_sim` | −1–1 | Más alto = perfil semántico más cercano |

El **score combinado** normaliza cada señal: `handle_sim + (1 - delta_norm) + embedding_sim` / 3.

In [ ]:
combined_df = pd.DataFrame()

# Build unified candidate pool
all_candidates = []

# From handle pivoting
if not attribution_candidates.empty:
    for _, row in attribution_candidates.iterrows():
        all_candidates.append({
            'user_a': f"{row['forum_a']}_{row.get('handle_a','')}",
            'user_b': f"{row['forum_b']}_{row.get('handle_b','')}",
            'handle_a': row.get('handle_a', ''),
            'handle_b': row.get('handle_b', ''),
            'forum_a': row['forum_a'],
            'forum_b': row['forum_b'],
            'handle_sim': row['handle_sim'],
            'embedding_sim': row['embedding_sim'],
            'delta': None,
        })

# From Delta cross-forum pairs
if not delta_pairs_df.empty:
    cross_only = delta_pairs_df[delta_pairs_df['cross_forum']].head(100)
    for _, row in cross_only.iterrows():
        all_candidates.append({
            'user_a': row['user_a'],
            'user_b': row['user_b'],
            'handle_a': row['user_a'].split('_', 1)[-1],
            'handle_b': row['user_b'].split('_', 1)[-1],
            'forum_a': row['forum_a'],
            'forum_b': row['forum_b'],
            'handle_sim': None,
            'embedding_sim': None,
            'delta': row['delta'],
        })

if all_candidates:
    cdf = pd.DataFrame(all_candidates)

    # Normalize delta (0=min → 1 score, max → 0 score)
    if cdf['delta'].notna().any():
        d_min = cdf['delta'].min()
        d_max = cdf['delta'].max()
        cdf['delta_norm'] = cdf['delta'].apply(
            lambda d: 0.0 if d is None else (1 - (d - d_min) / max(d_max - d_min, 1e-9))
        )
    else:
        cdf['delta_norm'] = np.nan

    # Compute combined score (only from available signals)
    def combined_score(row):
        signals = []
        if pd.notna(row.get('handle_sim')):
            signals.append(float(row['handle_sim']))
        if pd.notna(row.get('delta_norm')):
            signals.append(float(row['delta_norm']))
        if pd.notna(row.get('embedding_sim')):
            signals.append(float(row['embedding_sim']))
        return round(np.mean(signals), 4) if signals else np.nan

    cdf['combined_score'] = cdf.apply(combined_score, axis=1)
    combined_df = cdf.sort_values('combined_score', ascending=False).drop_duplicates(
        subset=['forum_a', 'forum_b', 'handle_a', 'handle_b']
    )

    print("Top 10 candidatos de atribución cross-foro:")
    print(
        combined_df[['handle_a', 'forum_a', 'handle_b', 'forum_b',
                      'handle_sim', 'delta', 'embedding_sim', 'combined_score']]
        .head(10)
        .to_string(index=False)
    )
else:
    print("Sin candidatos de atribución disponibles.")

## Resumen ejecutivo

Síntesis cuantitativa del análisis de embeddings y perfilado cross-foro.

In [ ]:
print("=" * 65)
print("HACKING FORUMS — 03: EMBEDDINGS Y PERFILADO — RESUMEN")
print("=" * 65)
print(f"Usuarios embed_users (A)      : {len(user_ids_a):,}")
print(f"Usuarios centroides (C)       : {len(user_ids_c):,}")

if spearman_rho is not None:
    print(f"Spearman ρ (A vs C)           : {spearman_rho:.4f}")
else:
    print("Spearman ρ (A vs C)           : N/A")

if len(hdb_labels) > 0:
    n_cl = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    print(f"Clusters HDBSCAN (C)          : {n_cl}")
else:
    print("Clusters HDBSCAN (C)          : N/A")

if not exact_matches_df.empty:
    print(f"Handles exactos cross-foro    : {len(exact_matches_df):,}")
else:
    print("Handles exactos cross-foro    : N/A")

if not fuzzy_results_df.empty:
    print(f"Matches fuzzy ≥85%            : {len(fuzzy_results_df):,}")
else:
    print("Matches fuzzy ≥85%            : N/A")

if not delta_pairs_df.empty:
    cross_count = delta_pairs_df['cross_forum'].sum()
    print(f"Pares Delta cross-foro        : {cross_count:,}")
else:
    print("Pares Delta cross-foro        : N/A")

if not combined_df.empty:
    print(f"Candidatos atribución final   : {len(combined_df):,}")
    top1 = combined_df.iloc[0]
    print(f"  Top candidato : {top1['handle_a']} ({top1['forum_a']}) ↔ "
          f"{top1['handle_b']} ({top1['forum_b']}) — score={top1['combined_score']}")
else:
    print("Candidatos atribución final   : N/A")

print("=" * 65)